# Davinci Notebook 11 — Evaluation Harness & Baselines

A practical framework for perplexity/loss evaluation, Persian text normalization diagnostics, task scoring, tool-call JSON validation, and reproducible test manifests. This is an evaluation scaffold; benchmark datasets require separate acquisition, licensing, and preprocessing.

In [ ]:
import torch, json, math, re, time, hashlib, unicodedata
from pathlib import Path


## 1. Text normalization and diagnostics

In [ ]:
def normalize_fa(s):
 s=unicodedata.normalize('NFC',str(s)).replace('ي','ی').replace('ى','ی').replace('ك','ک').replace('ـ','')
 s=re.sub('[\\u200e\\u200f\\u202a-\\u202e\\u2066-\\u2069]','',s); return s
def script_stats(text):
 return {'chars':len(text),'persian_arabic_block':sum('\\u0600'<=c<='\\u06ff' for c in text),'latin':sum(('a'<=c.lower()<='z') for c in text),'digits':sum(c.isdigit() for c in text)}


## 2. Model evaluation: average loss and perplexity
Requires a compatible model and tokenized batch loader. The model must return logits `[B,T,V]`.

In [ ]:
@torch.no_grad()
def evaluate_lm(model,loader,device,max_batches=None):
 model.eval(); total_loss=0.; total_tokens=0; batches=0
 for batch in loader:
  if max_batches and batches>=max_batches: break
  batch=batch.to(device); logits=model(batch); labels=batch[:,1:]; pred=logits[:,:-1,:].float()
  loss=torch.nn.functional.cross_entropy(pred.reshape(-1,pred.size(-1)),labels.reshape(-1),reduction='sum')
  total_loss+=loss.item(); total_tokens+=labels.numel(); batches+=1
 mean=total_loss/max(1,total_tokens); return {'mean_loss':mean,'perplexity':math.exp(min(80,mean)),'tokens':total_tokens,'batches':batches}


## 3. Tool-call format validation
Use a canonical JSON representation for generated calls; tool execution itself must be sandboxed and permission-controlled.

In [ ]:
def validate_tool_call(obj,available_tools):
 errors=[]
 if not isinstance(obj,dict): return ['Expected object']
 if obj.get('type')!='tool_call': errors.append('type must be tool_call')
 tool=obj.get('name'); args=obj.get('arguments')
 if tool not in available_tools: errors.append('unknown tool name')
 if not isinstance(args,dict): errors.append('arguments must be a JSON object')
 return errors
example={'type':'tool_call','name':'search','arguments':{'query':'example'}}
print('Example validation:',validate_tool_call(example,['search','browser']))


## 4. Benchmark run manifest
Record model revision, tokenizer revision, dataset identifiers, split, prompt format, sampling settings, and evaluator version for reproducibility.

In [ ]:
def write_eval_manifest(path,manifest):
 manifest=dict(manifest); manifest['created_utc']=time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()); manifest['manifest_sha256']=hashlib.sha256(json.dumps(manifest,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
 Path(path).write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8'); return manifest
sample_manifest={'model':'Davinci experimental','model_revision':'fill-in','tokenizer':'fill-in','dataset':'fill-in','split':'test','seed':42,'prompt_template_version':'v1'}
print('Fill the manifest fields with actual values; do not treat this placeholder as a benchmark result.')


## 5. Benchmarking checklist
- Persian language understanding: curated task sets and human review.
- English and bilingual capability: separately scored.
- Long-context retrieval: vary evidence location and distractors.
- Tool selection, arguments, execution outcomes, and recovery.
- Safety and refusal quality.
- Contamination checks and licensing/provenance audit.
- Report confidence intervals and multiple seeds where practical.
